# Day 03 — Exploratory Data Analysis & Descriptive Statistics

**Objective:** Understand a dataset statistically before building a machine-learning model.

We will study central tendency, dispersion, percentiles, distributions, skewness, kurtosis, outliers, covariance and correlation using Titanic passenger data.

**Dataset:** Titanic (Seaborn dataset, with a public CSV fallback).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid")

try:
    df = sns.load_dataset("titanic")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    raw = pd.read_csv(url)
    df = raw.rename(columns={"Survived":"survived", "Pclass":"pclass",
        "Sex":"sex", "Age":"age", "SibSp":"sibsp", "Parch":"parch",
        "Fare":"fare", "Embarked":"embarked"})

df = df.dropna(subset=["survived"]).reset_index(drop=True)
df.head()

## 1. Understand the dataset\nIdentify observations, variables and data types. A numeric storage type does not automatically make a variable continuous: passenger class is an ordered category.

In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
df.info()

## 2. Missing-value study

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing)
missing[missing.missing_count > 0]["missing_percent"].sort_values().plot(
    kind="barh", title="Missing values by column (%)", figsize=(8, 4))
plt.xlabel("Missing percentage")
plt.show()

## 3. Measures of central tendency

- **Mean:** arithmetic average; sensitive to extreme values.
- **Median:** middle value after sorting; more resistant to extremes.
- **Mode:** most frequent value; also useful for categorical data.

In [ ]:
numeric_cols = ["age", "fare", "sibsp", "parch"]
central = pd.DataFrame({
    "mean": df[numeric_cols].mean(),
    "median": df[numeric_cols].median(),
    "mode": df[numeric_cols].mode().iloc[0]
})
display(central)

values = pd.Series([25, 28, 30, 31, 32, 34, 35])
with_outlier = pd.concat([values, pd.Series([300])], ignore_index=True)
display(pd.DataFrame({
    "original": [values.mean(), values.median()],
    "with_extreme_value": [with_outlier.mean(), with_outlier.median()]
}, index=["mean", "median"]))

## 4. Measures of dispersion

- **Range:** maximum minus minimum.
- **Variance:** average squared deviation from the mean.
- **Standard deviation:** square root of variance, in the original unit.
- **IQR:** Q3 − Q1, the spread of the middle 50%.

Pandas uses `ddof=1` by default for sample variance and standard deviation; `ddof=0` calculates the population version.

In [ ]:
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)
dispersion = pd.DataFrame({
    "min": df[numeric_cols].min(),
    "max": df[numeric_cols].max(),
    "range": df[numeric_cols].max() - df[numeric_cols].min(),
    "sample_variance": df[numeric_cols].var(ddof=1),
    "sample_std": df[numeric_cols].std(ddof=1),
    "Q1": q1, "Q3": q3, "IQR": q3 - q1
})
display(dispersion)

display(pd.DataFrame({
    "sample_ddof_1": [df.age.var(ddof=1), df.age.std(ddof=1)],
    "population_ddof_0": [df.age.var(ddof=0), df.age.std(ddof=0)]
}, index=["variance", "standard_deviation"]))

## 5. Percentiles and quartiles

The 50th percentile is the median; the 25th and 75th percentiles are Q1 and Q3. Percentiles help describe where observations sit within a distribution.

In [ ]:
display(df[numeric_cols].quantile([0, .10, .25, .50, .75, .90, 1]))
print(f"90th percentile of fare: {df.fare.quantile(.90):.2f}")

## 6. Distribution analysis

Histograms show frequency across bins. KDE curves are smoothed density estimates; their total area is approximately one, unlike a count histogram.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, ax in zip(numeric_cols, axes.flat):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for col, ax in zip(["age", "fare"], axes):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.axvline(df[col].mean(), linestyle="--", label="Mean")
    ax.axvline(df[col].median(), linestyle="-.", label="Median")
    ax.legend()
    ax.set_title(f"{col}: mean vs median")
plt.tight_layout()
plt.show()

## 7. Skewness and kurtosis

**Skewness** describes asymmetry: positive skew has a longer right tail; negative skew has a longer left tail; values near zero are more symmetric.

**Excess kurtosis** compares tail behaviour with a normal distribution. Under this convention, a normal distribution has excess kurtosis near zero. It is not simply a measure of peakedness.

In [ ]:
display(pd.DataFrame({
    "skewness": df[numeric_cols].skew(),
    "excess_kurtosis": df[numeric_cols].kurt()
}))

rng = np.random.default_rng(42)
synthetic = {
    "Approximately symmetric": rng.normal(0, 1, 3000),
    "Right-skewed": rng.exponential(1, 3000),
    "Left-skewed": -rng.exponential(1, 3000)
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for (name, values), ax in zip(synthetic.items(), axes):
    sns.histplot(values, kde=True, ax=ax)
    ax.set_title(f"{name}\nSkew = {skew(values):.2f}")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    name: {"mean": np.mean(v), "median": np.median(v),
           "skewness": skew(v), "excess_kurtosis": kurtosis(v)}
    for name, v in synthetic.items()
}).T)

## 8. Outlier detection

An outlier may be an error, a rare but valid observation, or evidence of a different population. Do not automatically delete flagged observations.

**IQR fences:** lower = Q1 − 1.5 × IQR; upper = Q3 + 1.5 × IQR.

**Z-score:** z = (x − mean) / standard deviation. A common screening threshold is |z| > 3, but it is most interpretable for roughly normal distributions.

In [ ]:
def iqr_summary(series):
    s = series.dropna()
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = (s < low) | (s > high)
    return {"lower_fence": low, "upper_fence": high,
            "flagged_count": int(flagged.sum()),
            "flagged_percent": flagged.mean() * 100}

display(pd.DataFrame({c: iqr_summary(df[c]) for c in numeric_cols}).T)

z_results = {}
for col in numeric_cols:
    s = df[col].dropna()
    z = (s - s.mean()) / s.std(ddof=0)
    z_results[col] = {"abs_z_gt_3_count": int((z.abs() > 3).sum()),
                      "flagged_percent": (z.abs() > 3).mean() * 100}
display(pd.DataFrame(z_results).T)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for col, ax in zip(numeric_cols, axes):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 9. Covariance and correlation

- **Covariance** indicates whether two variables tend to move together; its magnitude depends on units.
- **Pearson correlation** ranges from −1 to +1 and measures linear association.
- Correlation does not establish causation. A near-zero Pearson correlation does not rule out nonlinear relationships.

In [ ]:
corr_cols = ["age", "fare", "sibsp", "parch", "survived"]
display(df[corr_cols].cov())
corr = df[corr_cols].corr()
display(corr)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Pearson correlation heatmap")
plt.show()

## 10. Categorical and grouped analysis

In [ ]:
for col in ["sex", "pclass", "embarked"]:
    if col in df.columns:
        print(f"Frequency table: {col}")
        display(df[col].value_counts(dropna=False).to_frame("count"))

display(df.groupby("sex", observed=True)["survived"].agg(
    count="count", survival_rate="mean"))
display(df.groupby("pclass")["survived"].agg(
    count="count", survival_rate="mean"))

## 11. Your statistical interpretation

Answer these after running the notebook:

1. Which variable has the largest spread? Does its unit affect the comparison?
2. How do mean and median compare for age and fare?
3. Which variables are positively or negatively skewed?
4. What does excess kurtosis suggest, and what are its limitations?
5. How many values are flagged by IQR and Z-score methods? Do the methods agree?
6. Which numerical variables have the strongest linear association?
7. What do survival rates by sex and class show? Why is this not proof of causation?
8. What important information is missing from a table of means and standard deviations?

## Key takeaways

- Mean and median describe centre but respond differently to extreme values.
- Variance and standard deviation measure spread; IQR describes the middle 50%.
- Percentiles describe position in a distribution.
- Histograms and density plots reveal distribution shape.
- Skewness describes asymmetry; excess kurtosis summarizes tail behaviour relative to a normal reference.
- IQR and Z-score rules flag unusual observations; neither proves an error.
- Correlation describes association, not causation.
- Interpret statistics in context: consider missingness, units, sampling and data collection.

## References

- [Pandas descriptive statistics](https://pandas.pydata.org/docs/user_guide/basics.html#descriptive-statistics)
- [Pandas quantile](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.quantile.html)
- [SciPy skew](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skew.html)
- [SciPy kurtosis](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kurtosis.html)
- [Titanic dataset](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv)

**Day 03 principle:** Don't just calculate statistics. Understand what they reveal, what they hide, and what conclusions the data cannot support.